# Hypothesis Testing with Chi-Square

# Introduction

In the previous lesson you identified the applicants who never complete
the admissions quiz, split them into control and treatment groups, and
wrote null and alternate hypotheses. You set the stage. **Now you run the
play.** This lesson takes the experiment from plan to verdict: you'll
decide how big the experiment must be, how long to run it, and — once the
data is in — whether the reminder email made a *real* difference or just a
lucky-looking one.

The engine for that final decision is the **chi-square test of
independence**, the standard tool for asking whether two categorical
variables (here: *group* and *quiz outcome*) are related.

> 🎯 **By the end of this notebook you will be able to:**
>
> - Perform a statistical power analysis to determine the required sample
>   size for an experiment.
> - Visualize the relationship between effect size, power, and sample
>   size using power curves.
> - Estimate experiment duration using the central limit theorem and
>   cumulative density functions.
> - Run a simulated A/B experiment and collect results from MongoDB.
> - Build contingency tables and visualize group-level outcomes.
> - Conduct a chi-square test of independence and interpret p-values.
> - Calculate and interpret odds ratios from a contingency table.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1183284881", h="3298dbabb7", width=700, height=450) 

# 1. Conceptual Foundation

## Project context: what we're trying to build

Across the previous lessons you explored applicant data in MongoDB and
built an ETL pipeline to prepare it. You found that a slice of applicants
never complete the admissions quiz, and you formed a hypothesis:
**a reminder email might raise the completion rate.**

This lesson is about the statistical machinery that separates **signal**
(a real effect) from **noise** (random variation). The workflow has three
acts:

| Act | Question it answers | Tool |
|---|---|---|
| **Plan** | How many participants do I need? | Power analysis |
| **Run** | Collect the data | `Experiment` simulator |
| **Evaluate** | Did the email actually matter? | Chi-square test |

The payoff is a principled, evidence-based answer to *"Did the email make
a difference?"* — not a hunch from eyeballing a bar chart.

## Statistical power and the four interlocking quantities

Before running *any* experiment, a careful analyst asks: *"How many
observations do I need?"* Too few and a real effect slips through
undetected; too many and you've burned time and money. The answer is
governed by **statistical power**.

🧠 **Intuition — power is an antenna.** A small antenna picks up only
strong signals (large effects) and misses faint ones. A bigger antenna
(more observations) can resolve weaker signals. "Power" is the
probability your test's antenna catches the signal *when there really is
one*.

Formally, every hypothesis test can make two kinds of mistakes, and the
four planning quantities map directly onto them:

| Quantity | Symbol | Meaning | Convention |
|---|---|---|---|
| Significance | $\alpha$ | P(false positive) — reject $H_0$ when it's true (**Type I error**) | 0.05 |
| Power | $1-\beta$ | P(detect a true effect) — where $\beta$ = P(**Type II error**) | 0.80 |
| Effect size | $w$ | How large the group difference is | — |
| Sample size | $n$ | Observations per group | solve for this |

🧮 **Power and the two error types.** Let $\beta$ be the probability of a
**Type II error** (missing a real effect). Then power is simply its
complement:

$$
\text{power} = 1 - \beta
$$

Setting power to 0.80 means: *if* the email truly works, we want an 80%
chance of detecting it. The two errors trade off against each other —
demanding a smaller $\alpha$ (fewer false alarms) pushes $\beta$ up
(more misses) unless you compensate with a larger sample.

🧮 **Effect size (Cohen's $w$).** For a chi-square test the effect size
measures how far the true cell proportions $p_{1i}$ sit from the
no-effect proportions $p_{0i}$:

$$
w = \sqrt{\sum_{i} \frac{(p_{1i} - p_{0i})^2}{p_{0i}}}
$$

By convention $w \approx 0.1$ is small, $0.3$ medium, and $0.5$ large.
A small effect needs a *big* sample to detect; a large effect shows up
even in a small one.

Given any **three** of {$\alpha$, power, $w$, $n$} you can solve for the
fourth. The `GofChisquarePower` class from statsmodels does the algebra.

### Demo: power calculation with `GofChisquarePower`

In [ ]:
import math
from statsmodels.stats.power import GofChisquarePower

# Instantiate the power analysis object
power_analysis = GofChisquarePower()

# Solve for sample size given effect_size, alpha, power
n = power_analysis.solve_power(
    effect_size=0.3, alpha=0.05, power=0.8
)
print("Raw n:", n)
print("Rounded up:", math.ceil(n))

The result is the number of observations needed **per group**. An A/B
experiment has two groups (control and treatment), so the total required
is roughly $2n$. Notice how the three knobs feed in: tighten $\alpha$,
raise the power target, or shrink the effect size you want to catch, and
the required $n$ climbs.

### Demo: plotting a power curve

A **power curve** plots power (y) against sample size (x) for a fixed
effect size. Overlaying several effect sizes shows, at a glance, how much
data each one demands to clear the 80% line.

In [ ]:
import numpy as np

# Range of sample sizes to evaluate
n_obs = np.arange(5, 300)

# Plot power curves for three effect sizes
power_analysis.plot_power(
    dep_var="nobs",
    nobs=n_obs,
    effect_size=np.array([0.2, 0.5, 0.8]),
    alpha=0.05,
    n_bins=2,
);

📊 **Read the curves.** Larger effect sizes (the upper curves) reach 80%
power with far fewer observations; small effects (the lower curve) need a
much bigger sample to climb the same height. This is why a *realistic*
guess of your effect size — made **before** the experiment — is so
important: assume a bigger effect than is real and you'll under-recruit
and miss it.

## Estimating experiment duration with the CLT

Knowing you need $2n$ participants is only half the plan — you also need
to know *how long* recruiting them will take. If you know the daily
arrival rate of eligible (no-quiz) applicants, the **central limit
theorem (CLT)** lets you model the running total.

🧮 The CLT says the sum of many independent daily counts is approximately
**normal**, regardless of the shape of a single day's distribution. If
daily arrivals have mean $\mu$ and standard deviation $\sigma$, then the
total over $d$ days is approximately:

$$
\text{Total} \sim N\!\left(d \cdot \mu,\; \sigma\sqrt{d}\right)
$$

Note the asymmetry: the **mean** grows linearly with $d$ (multiply by
$d$), but the **standard deviation** grows only with $\sqrt{d}$. So the
longer you run, the more the predictable signal ($d\mu$) dominates the
random spread ($\sigma\sqrt{d}$) — recruitment totals get *relatively*
more predictable over time.

With that normal model in hand, the **cumulative density function (CDF)**
gives the probability of having reached your target by day $d$.

### Demo: CDF calculation with `scipy.stats`

In [ ]:
import scipy.stats

# Suppose daily arrivals: mean=40, std=8
daily_mean, daily_std = 40, 8
days = 10
target = 400

# Sum distribution over `days` days
sum_mean = daily_mean * days
sum_std = daily_std * np.sqrt(days)

# P(Total >= target) = 1 - CDF(target)
p_reach = 1 - scipy.stats.norm.cdf(
    target, loc=sum_mean, scale=sum_std
)
print(f"P(>= {target} in {days} days): {p_reach:.3f}")

The CDF gives $P(\text{Total} \le \text{target})$, so $1 - \text{CDF}$ is
the probability of reaching the target **or more**. A value near 0.90 or
above gives reasonable confidence you'll collect enough data within the
planned window.

## Contingency tables and the chi-square test

After the experiment you compare outcomes across groups with a
**contingency table** — a matrix cross-tabulating two categorical
variables (group × outcome):

|  | Completed | Not completed | Row total |
|---|---|---|---|
| **Control** | $O_{11}$ | $O_{12}$ | $R_1$ |
| **Treatment** | $O_{21}$ | $O_{22}$ | $R_2$ |
| **Col total** | $C_1$ | $C_2$ | $N$ |

The **chi-square test of independence** asks whether the two variables
are associated:

- $H_0$: group and outcome are **independent** (the email has no effect).
- $H_1$: group and outcome are **associated** (the email matters).

🧮 **The logic in three steps.**

**1. Expected counts.** *If* $H_0$ were true (independence), the count in
each cell would depend only on the row and column totals:

$$
E_{ij} = \frac{R_i \, C_j}{N}
$$

This is just "what each cell *should* hold if the outcome rate were
identical across groups."

**2. The test statistic.** Sum the squared gaps between what you
**observed** ($O_{ij}$) and what independence **expects** ($E_{ij}$),
each scaled by the expected count:

$$
\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}
$$

Big observed-minus-expected gaps → big $\chi^2$ → strong evidence against
$H_0$.

**3. Degrees of freedom and the p-value.** The statistic is compared to a
chi-square distribution with

$$
\text{df} = (r-1)(c-1)
$$

degrees of freedom — for a 2×2 table that's $(2-1)(2-1) = 1$. The
**p-value** is the probability of seeing a $\chi^2$ at least this large
*if $H_0$ were true*. A small p-value (≤ $\alpha$) means the data would be
surprising under independence, so we reject $H_0$.

### Demo: contingency table and chi-square test

In [ ]:
import pandas as pd
from statsmodels.stats.contingency_tables import Table2x2

# Toy example: 2×2 observed counts
#             Completed  Not Completed
# Control        30          70
# Treatment      45          55
toy_data = pd.DataFrame(
    {"complete": [30, 45], "incomplete": [70, 55]},
    index=["control", "treatment"],
)
print(toy_data)

The `Table2x2` object wraps the raw counts and exposes the machinery
above. Its `fittedvalues` are exactly the expected counts $E_{ij}$ from
the formula:

In [ ]:
# Wrap in a Table2x2 object
table = Table2x2(toy_data.values)

# Expected counts under independence
print("Fitted (expected) values:")
print(table.fittedvalues.round(1))

Compare these fitted (expected) values to the toy observed counts: the
larger the gap, the more the data departs from independence. The next
cell turns that gap into a single $\chi^2$ statistic and p-value.

In [ ]:
# Run chi-square test
test_result = table.test_nominal_association()
print(test_result)

🧮 **The odds ratio.** Beyond *whether* there's an association, the
**odds ratio (OR)** says *how strong* it is. For a 2×2 table

$$
\text{OR} = \frac{O_{11}\,O_{22}}{O_{12}\,O_{21}} = \frac{ad}{bc}
$$

it compares the *odds* of completing the quiz in one group to the odds in
the other. OR = 1 means no difference; OR > 1 means the treatment group
has higher odds of completing; OR < 1 means lower.

In [ ]:
# Odds ratio: how much more likely is the treatment group
# to complete vs. control?
print("Odds ratio:", table.oddsratio.round(2))

Putting it together: if the p-value is below your $\alpha$ (0.05), you
**reject** $H_0$ and conclude there is a statistically significant
association. The odds ratio then quantifies how much more (or less) likely
one group is to achieve the outcome.

### Practical vs. statistical significance

⚠️ **A significant result is not automatically an important one.** With a
huge sample, even a trivial 0.5% lift in completion can be "statistically
significant" yet far too small to justify the cost of an email campaign.
Conversely a large, business-relevant effect can fail to reach
significance if the sample is small. Always read the **p-value** (is the
effect real?) *and* the **effect size / odds ratio** (is it big enough to
act on?) together.

### Common pitfalls

- **Peeking / stopping early:** checking results before the planned
  sample is collected inflates the false-positive rate — you keep looking
  until noise crosses the line.
- **Multiple comparisons:** testing many hypotheses on the same data
  raises the chance that *something* looks significant by luck.
- **Correlation ≠ causation:** a significant $\chi^2$ shows *association*.
  It only supports a causal claim because the experiment was randomised —
  randomisation is what licenses the leap from "associated" to "caused."

## The `Experiment` helper class

Running a real A/B test over many days would take weeks. So you can
practise the full workflow in one session, the project ships a simulator:
the `Experiment` class from `wqulibs.ab_test.experiment`.

It generates synthetic applicants day by day, randomly assigns each to
control (`"no email (control)"`) or treatment (`"email (treatment)"`),
and inserts the documents into your MongoDB collection. You don't modify
it — just import it and call three methods:

- `Experiment(repo=client, db="wqu-abtest", collection="ds-applicants")`
  — create an instance connected to your database.
- `exp.reset_experiment()` — clear documents from a previous run.
- `exp.run_experiment(days=n)` — simulate `n` days of the experiment.

Afterwards the new documents live in MongoDB; query them with
`find({"inExperiment": True})` to build the contingency table and run the
test.

> 📚 **Key references for this lesson:**
>
> - [`GofChisquarePower`](https://www.statsmodels.org/stable/generated/statsmodels.stats.power.GofChisquarePower.html)
>   — power analysis for chi-square goodness-of-fit tests.
> - [`scipy.stats.norm.cdf`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html)
>   — cumulative density function for the normal distribution.
> - [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)
>   — compute a cross-tabulation of two categorical variables.
> - [`Table2x2`](https://www.statsmodels.org/stable/generated/statsmodels.stats.contingency_tables.Table2x2.html)
>   — contingency table analysis including chi-square tests and odds
>   ratios.

# Applied Exercises

## 2. Setup

Note: Import the required libraries and classes. It is highly recommended
to place all imports in a single cell at the beginning of the notebook.

**Code 7.3.2.1**:

In [ ]:
!pip install -q names randomtimestamp

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import scipy
from pymongo import MongoClient
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.power import GofChisquarePower
from wqulibs.ab_test.experiment import Experiment
from wqulibs.database import reset

## 3. Connect to MongoDB and Calculate Power

### Problem

Before running the experiment, you need to connect to the applicant
database and determine how many participants are required. Without a
proper power analysis, you risk either wasting resources on an overpowered
study or failing to detect a real effect with an underpowered one.

### Approach

You will connect to the `"wqu-abtest"` database and access the
`"ds-applicants"` collection using `MongoClient`. Then you will use
`GofChisquarePower` to solve for the required group size given an effect
size of 0.2, alpha of 0.05, and power of 0.8. Finally, you will plot power
curves for three effect sizes (0.2, 0.5, 0.8) to visualize how sample size
requirements change.

### Tasks

First, reset the database and connect to the `"ds-applicants"`
collection. Assign it to `ds_app`.

**🛠️ Instruction:** Locate the IP address of the machine running MongoDB
and assign it to the variable `MONGODB_HOST`. Make sure to use a
**string** (i.e., wrap the IP in quotes).

**⚠️ Note:** The IP address is **dynamic** — it may change every time you
start the lab. Always check the current IP before proceeding.

<figure>
<img src="attachment:images/mongo_ip.png" alt="MongoDB" />
<figcaption aria-hidden="true">MongoDB</figcaption>
</figure>

**Code 7.3.3.1**:

In [ ]:
MONGODB_HOST = "localhost"

client = MongoClient(host=MONGODB_HOST, port=27017)
ds_app = client["wqu-abtest"]["ds-applicants"]
reset(ds_app)
print("client:", type(client))
print("ds_app:", type(ds_app))

Instantiate a `GofChisquarePower` object and calculate the required
`group_size` to detect an effect size of 0.2 with alpha 0.05 and power
0.8. Use `math.ceil` to round up.

💡 Note the deliberately small effect size (0.2): you're asking the test
to catch even a *modest* lift in completion, which is why — per the power
formula — the required sample comes out in the hundreds.

**Code Task 7.3.3.2**:

In [ ]:
chi_square_power = GofChisquarePower()  # instantiate GofChisquarePower
group_size = math.ceil(
    chi_square_power.solve_power(
        effect_size=0.2, alpha=0.05, power=0.8
    )
)
print("Group size:", group_size)
print("Total # of applicants needed:", group_size * 2)

📊 The printout shows the per-group size and double it for the total. With
$w = 0.2$, $\alpha = 0.05$, and power $= 0.80$, expect a group size in the
low-to-mid hundreds — the price of being able to detect a small effect.

Now plot power curves for effect sizes 0.2, 0.5, and 0.8. The x-axis
should range from 0 to `2 * group_size` observations.

**Code 7.3.3.3**:

In [ ]:
n_observations = np.arange(0, group_size * 2) # range up to group_size * 2
effect_sizes = np.array([0.2, 0.5, 0.8])
chi_square_power.plot_power(
    dep_var="nobs",
    nobs=n_observations,
    effect_size=effect_sizes,
    alpha=0.05,
    n_bins=2,
);

📊 **Interpretation.** The three curves rise from left (few observations,
low power) to right. At any fixed sample size the large-effect curve
(0.8) sits highest and the small-effect curve (0.2) lowest. Your chosen
$2 \times$ `group_size` is exactly the x-position where the 0.2 curve
crosses 0.80 power — confirming the sample size you solved for.

### Checkpoint

In [ ]:
assert isinstance(ds_app, type(client["wqu-abtest"]["ds-applicants"])), (
    f"Expected ds_app to be a pymongo Collection, got {type(ds_app)}"
)
assert isinstance(group_size, int), (
    f"Expected group_size to be int, got {type(group_size)}"
)
assert 150 < group_size < 300, (
    f"Expected group_size between 150 and 300, got {group_size}"
)
print("✓ All checks passed.")

## 4. Estimate Experiment Duration

### Problem

You know the total number of participants required, but you need to figure
out how many days to run the experiment. The experiment targets applicants
who don't complete the admissions quiz, so you need to estimate the daily
arrival rate of that subset and then calculate the probability of reaching
your target within a given timeframe.

### Approach

You will use a MongoDB aggregation pipeline to count new incomplete-quiz
accounts per day. The result will be loaded into a pandas Series called
`no_quiz`. From the mean and standard deviation of daily arrivals, you
will apply the central limit theorem to estimate the probability of
reaching `group_size * 2` total participants within a chosen number of
days.

➡️ This section turns the abstract CLT formula
$\text{Total} \sim N(d\mu,\, \sigma\sqrt{d})$ from section 1 into a
concrete recruitment forecast using the database's own arrival history.

### Tasks

Use the `aggregate` method on `ds_app` to count the number of new accounts
with `"admissionsQuiz": "incomplete"` created each day. Group by date using
`$dateTrunc`.

**Reminder:** The result of `aggregate()` is a **cursor** — a single-use
stream. Once you iterate through it (e.g. with a `for` loop or `list()`),
the data is consumed. If you need the data again, re-run the `aggregate()`
call.

**Code 7.3.4.1**:

In [ ]:
result = ds_app.aggregate(
    [
        {"$match": {"admissionsQuiz": "incomplete"}},
        {
            "$group": {
                "_id": {
                    "$dateTrunc": {
                        "date": "$createdAt",
                        "unit": "day",
                    }
                },
                "count": {"$sum": 1},
            }
        },
    ]
)
print("result type:", type(result))

Read the aggregation `result` into a Series named `no_quiz`. Rename the
fields so the index is `"date"` and the values are `"new_users"`, then
sort by date.

**Code Task 7.3.4.2**:

In [ ]:
no_quiz = (
    pd.DataFrame(list(result))
    .rename({"_id": "date", "count": "new_users"}, axis=1)  # rename fields
    .set_index("date")
    .sort_index()
    .squeeze()
)
print("no_quiz type:", type(no_quiz))
print("no_quiz shape:", no_quiz.shape)
no_quiz.head()

Create a histogram of `no_quiz` to see the distribution of daily arrivals.
Label the axes and add a descriptive title.

**Code 7.3.4.3**:

In [ ]:
no_quiz.hist()
plt.xlabel("New Users with No Quiz")
plt.ylabel("Frequency [count]")
plt.title("Distribution of Daily New Users with No Quiz");

📊 The histogram shows roughly how many no-quiz applicants arrive on a
typical day and how much that count bounces around. It's this **mean** and
**spread** that feed the CLT — a tighter distribution makes recruitment
totals more predictable.

Calculate the `mean` and `std` of the `no_quiz` Series. These statistics
describe the daily arrival distribution.

**Code 7.3.4.4**:

In [ ]:
mean = no_quiz.mean()
std = no_quiz.std()
print("no_quiz mean:", mean)
print("no_quiz std:", std)

These are the $\mu$ (mean) and $\sigma$ (standard deviation) of daily
arrivals — the two inputs the CLT needs to model the multi-day total.

Using the CLT, calculate the mean and standard deviation of the total
sign-ups over 10 days. Then compute the probability of reaching
`group_size * 2` or more participants in that period.

**Code Task 7.3.4.5**:

In [ ]:
days = 10
# CLT: mean and std of the sum over days
sum_mean = mean * days
sum_std = std * np.sqrt(days)
print("Mean of sum:", sum_mean)
print("Std of sum:", sum_std)

# P(Total >= target) using the normal CDF
prob_400_or_fewer = scipy.stats.norm.cdf(
    group_size * 2,  # target number of participants
    loc=sum_mean,
    scale=sum_std
)
prob_400_or_greater = 1 - prob_400_or_fewer
print(
    f"P(>= {group_size * 2} in {days} days):",
    round(prob_400_or_greater, 3),
)

**Solution 7.3.4.5**:

In [ ]:
days = 10
sum_mean = mean * days
sum_std = std * np.sqrt(days)
print("Mean of sum:", sum_mean)
print("Std of sum:", sum_std)

# P(Total >= target)
prob_400_or_fewer = scipy.stats.norm.cdf(
    group_size * 2, loc=sum_mean, scale=sum_std
)
prob_400_or_greater = 1 - prob_400_or_fewer
print(
    f"P(>= {group_size * 2} in {days} days):",
    round(prob_400_or_greater, 3),
)

🔍 **What you just computed.** `sum_mean` $= \mu \cdot d$ and `sum_std`
$= \sigma\sqrt{d}$ are the parameters of the 10-day total's normal
distribution. `scipy.stats.norm.cdf(target, ...)` gives
$P(\text{Total} \le \text{target})$, so subtracting from 1 gives the
probability of hitting your target **or more**.

> 📌 **Tip:** The exact probability depends on your database, but you
> should see roughly a 90% chance of reaching enough participants in 10
> days. Try changing `days` to watch the probability shift — recall that
> the spread grows only as $\sqrt{d}$, so adding days raises the expected
> total faster than the uncertainty, pushing the probability up.

### Checkpoint

In [ ]:
assert isinstance(no_quiz, pd.Series), (
    f"Expected no_quiz to be a pandas Series, got {type(no_quiz)}"
)
assert no_quiz.shape[0] > 0, "no_quiz should not be empty."
assert 20 < mean < 80, (
    f"Expected daily mean between 20 and 80, got {mean:.1f}"
)
assert 0 < prob_400_or_greater <= 1.0, (
    f"Probability must be in (0, 1], got {prob_400_or_greater:.3f}"
)
print("✓ All checks passed.")

## 5. Run the Experiment

### Problem

With the power analysis complete and the duration estimated, it is time to
actually run the simulated A/B experiment. The `Experiment` class handles
the random assignment of applicants to control (no email) and treatment
(email) groups over the specified number of days.

### Approach

You will create an `Experiment` object connected to the MongoDB database,
reset any prior experiment state, and run the experiment for the number of
`days` determined in the previous section. The result is a summary
dictionary showing how many applicants were assigned to each group.

### Tasks

Create an `Experiment` instance, reset it, and run for the planned number
of days. The returned `result` is a summary of the experiment.

**Code 7.3.5.1**:

In [ ]:
exp = Experiment(
    repo=client,
    db="wqu-abtest",
    collection="ds-applicants",
)
exp.reset_experiment()
result = exp.run_experiment(days=days)
print("result type:", type(result))
result

✅ The summary reports how many applicants landed in each group. Because
`reset_experiment()` ran first, this is a clean run — no leftover
documents from a previous attempt skew the counts.

### Checkpoint

In [ ]:
assert result is not None, "Experiment result should not be None."
print("✓ Experiment completed successfully.")

## 6. Evaluate Experiment Results

### Problem

The experiment has been run. Now you need to retrieve the experimental
data from MongoDB, build a contingency table, visualize the results, and
conduct a chi-square test to determine whether the email intervention had
a statistically significant effect on quiz completion.

### Approach

You will query all documents with `"inExperiment": True`, load them into a
DataFrame, create a cross-tabulation of group × quiz completion using
`pd.crosstab`, visualize the result with a grouped bar chart, then wrap the
counts in a `Table2x2` object to perform the chi-square test and compute
the odds ratio.

➡️ This is the **Evaluate** act: everything from section 1's chi-square
logic ($E_{ij}$, $\chi^2$, df, OR) now gets applied to the real
experimental counts.

### Tasks

Query `ds_app` for all documents where `"inExperiment"` is `True`.

**Code 7.3.6.1**:

In [ ]:
# query all documents that are part of the experiment
result = ds_app.find({"inExperiment": True})
print("result type:", type(result))

Load the query `result` into a DataFrame `df` and drop any rows with `NaN`
values.

**Code 7.3.6.2**:

In [ ]:
df = pd.DataFrame(result).dropna()
print("df type:", type(df))
print("df shape:", df.shape)
df.head()

Use `pd.crosstab` to create a 2×2 table called `data` that cross-tabulates
`df["group"]` against `df["admissionsQuiz"]`.

🧱 This `data` table *is* the contingency table from section 1 — its four
cells are the observed counts $O_{ij}$ that the chi-square test compares
against the expected counts $E_{ij}$.

**Code Task 7.3.6.3**:

In [ ]:
data = pd.crosstab(
    index=df["group"],  # the group column
    columns=df["admissionsQuiz"],  # the quiz completion column
    normalize=False,
)
print("data type:", type(data))
print("data shape:", data.shape)
data

**Solution 7.3.6.3**:

In [ ]:
data = pd.crosstab(
    index=df["group"],
    columns=df["admissionsQuiz"],
    normalize=False,
)
print("data type:", type(data))
print("data shape:", data.shape)
data

Create a side-by-side bar chart from `data` using Plotly Express to
visualize quiz completion by group. Label the x-axis `"Group"`, y-axis
`"Frequency [count]"`, and use the title
`"Admissions Quiz Completion by Group"`.

**Code 7.3.6.4**:

In [ ]:
fig = px.bar(
    data_frame=data,
    barmode="group",
    title="Admissions Quiz Completion by Group",
)
fig.update_layout(
    xaxis_title="Group",
    yaxis_title="Frequency [count]",
    legend={"title": "Admissions Quiz"},
)
fig.show()

⚠️ **Don't trust your eyes yet.** Even if the treatment bars look taller,
the gap could be random noise — exactly the trap the chi-square test
exists to avoid. A visual difference is a *hypothesis*, not a conclusion.
The formal test comes next.

Wrap `data` in a `Table2x2` object called `contingency_table` and inspect
the observed counts.

**Code 7.3.6.5**:

In [ ]:
contingency_table = Table2x2(data.values)
print("contingency_table type:", type(contingency_table))
contingency_table.table_orig

Examine the expected (fitted) values under the assumption of independence,
and the joint independence probabilities.

🧮 The `fittedvalues` are the expected counts $E_{ij} = R_i C_j / N$ — what
each cell *would* hold if the email made no difference. The chi-square
statistic is built from how far the observed counts stray from these.

**Code 7.3.6.6**:

In [ ]:
# Expected counts under independence
print("Fitted values:")
print(contingency_table.fittedvalues.round())
print()
# Joint probabilities under independence
print("Independence probabilities:")
print(
    contingency_table.independence_probabilities.round(2)
)

📊 Compare `fittedvalues` (expected under $H_0$) to the observed counts in
`table_orig`. The wider the gap — especially in the treatment/completed
cell — the larger the $\chi^2 = \sum (O_{ij}-E_{ij})^2 / E_{ij}$ the test
will produce, and the smaller the p-value.

Now perform the chi-square test of independence and assign the result to
`chi_square_test`.

**Code Task 7.3.6.7**:

In [ ]:
# run the chi-square test of independence
chi_square_test = (
    contingency_table.test_nominal_association()
)
print("chi_square_test type:", type(chi_square_test))
print(chi_square_test)

**Solution 7.3.6.7**:

In [ ]:
chi_square_test = (
    contingency_table.test_nominal_association()
)
print("chi_square_test type:", type(chi_square_test))
print(chi_square_test)

🔍 **Reading the output.** The key number is the **p-value**, evaluated
against $\text{df} = (2-1)(2-1) = 1$ for this 2×2 table:

- **p ≤ 0.05** → reject $H_0$: the email had a statistically significant
  effect on quiz completion.
- **p > 0.05** → fail to reject $H_0$: the observed difference is
  plausibly just chance.

Because this is a *simulated* experiment, your p-value will vary run to
run. A non-significant result means the signal is too weak to detect at
this sample size (or there's no real effect); re-running with a longer
duration (e.g. `days=60`) raises power and can change the verdict.

Calculate the odds ratio and print the full summary of the contingency
table.

**Code 7.3.6.8**:

In [ ]:
odds_ratio = contingency_table.oddsratio.round(2)
print("Odds ratio:", odds_ratio)
print()
# Full summary
summary = contingency_table.summary()
print("summary type:", type(summary))
summary

🧮 **Interpreting the odds ratio** $\text{OR} = ad/bc$. An OR of, say,
1.3 means the treatment group's *odds* of completing the quiz are about
1.3× the control group's. OR = 1 would mean no difference. Crucially, the
odds ratio is only **actionable when the chi-square test is significant** —
otherwise you can't rule out that the "1.3" is noise. And even a
significant OR must clear the *practical* bar from section 1: is the lift
big enough to justify the campaign's cost?

### Checkpoint

In [ ]:
assert isinstance(data, pd.DataFrame), (
    f"Expected data to be a DataFrame, got {type(data)}"
)
assert data.shape == (2, 2), (
    f"Expected data shape (2, 2), got {data.shape}"
)
assert isinstance(contingency_table, Table2x2), (
    f"Expected Table2x2, got {type(contingency_table)}"
)
assert chi_square_test is not None, (
    "chi_square_test should not be None."
)
assert isinstance(odds_ratio, float), (
    f"Expected odds_ratio to be float, got {type(odds_ratio)}"
)
print(f"✓ Contingency table shape: {data.shape}")
print(f"✓ Odds ratio: {odds_ratio}")
print("✓ All checks passed.")

# Wrap-up

In this lesson you:

-   Connected to MongoDB and accessed the applicant collection for
    experimentation.
-   Performed a power analysis ($\alpha$, power $= 1-\beta$, effect size
    $w$, sample size $n$) to determine the required sample size for a
    chi-square test.
-   Visualized power curves showing how sample size requirements vary with
    effect size.
-   Estimated experiment duration using the central limit theorem
    ($\text{Total} \sim N(d\mu,\, \sigma\sqrt{d})$) and CDF calculations.
-   Ran a simulated A/B experiment assigning applicants to control and
    treatment groups.
-   Built a contingency table and visualized group-level quiz completion
    with a bar chart.
-   Conducted a chi-square test of independence
    ($\chi^2 = \sum (O_{ij}-E_{ij})^2 / E_{ij}$) and interpreted the
    p-value.
-   Calculated the odds ratio ($\text{OR} = ad/bc$) to quantify the
    relative likelihood of quiz completion between groups.

➡️ **Where this is heading:** you now have a tested, quantified result.
Next you'll wrap this whole workflow in an interactive **dashboard** so
stakeholders can monitor the experiment and read its findings without
touching a line of code.